In [1]:
import pandas as pd
from itertools import combinations

In [5]:
input_file = "Ground_truth_OBD_dependency.xlsx"

a_gt = pd.read_excel(input_file, sheet_name="A_GT")

rows = []

row_id = 1

# Group activities by user story
activities_by_story = {
    us_id: group.reset_index(drop=True)
    for us_id, group in a_gt.groupby("US_ID", sort=False)
}

story_ids = list(activities_by_story.keys())

for story_a_id, story_b_id in combinations(story_ids, 2):

    story_a = activities_by_story[story_a_id]
    story_b = activities_by_story[story_b_id]

    # Cartesian product between activities of the two stories
    for _, activity_a in story_a.iterrows():
        for _, activity_b in story_b.iterrows():

            rows.append({
                "row_id": row_id,

                "story_a_id": activity_a["US_ID"],
                "action_a": activity_a["US_Action"],
                "entity_a": activity_a["US_Entity"],
                "crud_a": activity_a["GT_operation_type"],

                "story_b_id": activity_b["US_ID"],
                "action_b": activity_b["US_Action"],
                "entity_b": activity_b["US_Entity"],
                "crud_b": activity_b["GT_operation_type"],

                # These columns are intentionally empty.
                # They are labels to be filled in later.
                "nlp_match_type (exact/semantic/none)": pd.NA,
                "gpt5_match_type (exact/semantic/none)": pd.NA,
                "gpt5_explanation": pd.NA,
                "human_final_match_type (exact/semantic/none)": pd.NA,
            })

            row_id += 1

b_gt_generated = pd.DataFrame(rows)


print(f"A_GT activities : {len(a_gt)}")
print(f"User stories    : {a_gt['US_ID'].nunique()}")
print(f"B_GT rows       : {len(b_gt_generated)}")

# Expected:
# A_GT activities : 34
# User stories    : 20
# B_GT rows       : 542

display(b_gt_generated.head(10))

A_GT activities : 34
User stories    : 20
B_GT rows       : 542


,row_id,story_a_id,action_a,entity_a,crud_a,story_b_id,action_b,entity_b,crud_b,nlp_match_type (exact/semantic/none),gpt5_match_type (exact/semantic/none),gpt5_explanation,human_final_match_type (exact/semantic/none)
0,1,US-001,sign up,email,CREATE,US-002,log in,account,READ,<NA>,<NA>,<NA>,<NA>
1,2,US-001,sign up,email,CREATE,US-002,log out,account,NON_CRUD,<NA>,<NA>,<NA>,<NA>
2,3,US-001,sign up,existing Google account,CREATE,US-002,log in,account,READ,<NA>,<NA>,<NA>,<NA>
3,4,US-001,sign up,existing Google account,CREATE,US-002,log out,account,NON_CRUD,<NA>,<NA>,<NA>,<NA>
4,5,US-001,sign up,existing Apple account,CREATE,US-002,log in,account,READ,<NA>,<NA>,<NA>,<NA>
5,6,US-001,sign up,existing Apple account,CREATE,US-002,log out,account,NON_CRUD,<NA>,<NA>,<NA>,<NA>
6,7,US-001,sign up,email,CREATE,US-003,walk through,"short, guided tutorial",READ,<NA>,<NA>,<NA>,<NA>
7,8,US-001,sign up,existing Google account,CREATE,US-003,walk through,"short, guided tutorial",READ,<NA>,<NA>,<NA>,<NA>
8,9,US-001,sign up,existing Apple account,CREATE,US-003,walk through,"short, guided tutorial",READ,<NA>,<NA>,<NA>,<NA>
9,10,US-001,sign up,email,CREATE,US-004,import,location,CREATE,<NA>,<NA>,<NA>,<NA>


In [6]:
import spacy

nlp = spacy.load("en_core_web_sm")
QUALIFIER_POS_TAGS = {"ADJ", "ADP", "ADV", "AUX", "CCONJ", "DET", "INTJ", "PUNCT", "SCONJ", "X"}

In [8]:
STATE_QUALIFIER_POS = {"ADJ"}          # e.g. "active", "signed" (when tagged ADJ)
STATE_QUALIFIER_TAGS = {"VBN", "VBG"}  # participles, e.g. "submitted", "existing"

STRUCTURAL_POS = {"DET", "PUNCT"}      # always stripped, never carries meaning


def clean_phrase(text: str, strip_state_qualifiers: bool = False) -> str:
    """
    Lemmatize each token, dropping structural words (DET/PUNCT) always,
    and optionally dropping state-describing qualifiers (ADJ / participles).
    Type qualifiers (NOUN/PROPN modifiers, e.g. "Google", "credit") are
    never stripped - they must remain to block false semantic matches.
    """
    doc = nlp(text)
    kept_lemmas = []

    for token in doc:
        if token.pos_ in STRUCTURAL_POS:
            continue
        if strip_state_qualifiers and (
            token.pos_ in STATE_QUALIFIER_POS
            or token.tag_ in STATE_QUALIFIER_TAGS
        ):
            continue
        kept_lemmas.append(token.lemma_.lower())

    return " ".join(kept_lemmas)


def is_exact_match(phrase_a: str, phrase_b: str) -> bool:
    """
    1.1 - Same lexical item, differing only by plural/singular,
    case/punctuation, or determiner - no descriptive content added/removed.
    """
    return clean_phrase(phrase_a) == clean_phrase(phrase_b)


def is_semantic_match(phrase_a: str, phrase_b: str) -> bool:
    """
    1.2 - Same referent, but one phrase carries an extra state/condition
    qualifier (e.g. "submitted", "signed", "existing"). Type qualifiers
    (e.g. "Google", "credit") are intentionally NOT stripped, so pairs
    like "account" / "Google account" will NOT match here - deferred to LLM.
    """
    if is_exact_match(phrase_a, phrase_b):
        return False  # already classified as exact, avoid double-counting

    return clean_phrase(phrase_a, strip_state_qualifiers=True) == \
           clean_phrase(phrase_b, strip_state_qualifiers=True)


def get_nlp_match_type(entity_a, entity_b) -> str:
    if pd.isna(entity_a) or pd.isna(entity_b):
        return "none"

    entity_a = str(entity_a)
    entity_b = str(entity_b)

    if is_exact_match(entity_a, entity_b):
        return "exact"

    if is_semantic_match(entity_a, entity_b):
        return "semantic"

    return "none"

In [10]:
b_gt_generated["nlp_match_type (exact/semantic/none)"] = b_gt_generated.apply(
    lambda row: get_nlp_match_type(
        row["entity_a"],
        row["entity_b"]
    ),
    axis=1
)


display(
    b_gt_generated[
        [
            "story_a_id",
            "entity_a",
            "story_b_id",
            "entity_b",
            "nlp_match_type (exact/semantic/none)"
        ]
    ].head(20)
)


print("\nNLP lexical/semantic match summary:")
print(
    b_gt_generated["nlp_match_type (exact/semantic/none)"]
    .value_counts(dropna=False)
)

# Show only the rows classified as exact or semantic
matched_rows = b_gt_generated[
    b_gt_generated["nlp_match_type (exact/semantic/none)"].isin(["exact", "semantic"])
]

print(f"\nMatched rows (exact + semantic): {len(matched_rows)}")
display(
    matched_rows[
        [
            "story_a_id",
            "entity_a",
            "story_b_id",
            "entity_b",
            "nlp_match_type (exact/semantic/none)",
        ]
    ]
)


,story_a_id,entity_a,story_b_id,entity_b,nlp_match_type (exact/semantic/none)
0,US-001,email,US-002,account,none
1,US-001,email,US-002,account,none
2,US-001,existing Google account,US-002,account,none
3,US-001,existing Google account,US-002,account,none
4,US-001,existing Apple account,US-002,account,none
5,US-001,existing Apple account,US-002,account,none
6,US-001,email,US-003,"short, guided tutorial",none
7,US-001,existing Google account,US-003,"short, guided tutorial",none
8,US-001,existing Apple account,US-003,"short, guided tutorial",none
9,US-001,email,US-004,location,none



NLP lexical/semantic match summary:
nlp_match_type (exact/semantic/none)
none        534
exact         6
semantic      2
Name: count, dtype: int64

Matched rows (exact + semantic): 8


,story_a_id,entity_a,story_b_id,entity_b,nlp_match_type (exact/semantic/none)
187,US-004,location,US-009,location,exact
213,US-005,saved spots,US-008,saved spot,exact
223,US-005,saved spots,US-011,saved spots,exact
234,US-005,named lists,US-014,list,semantic
328,US-008,saved spot,US-011,saved spots,exact
346,US-009,name,US-010,name,exact
383,US-009,spot,US-017,spots,exact
517,US-015,generated itinerary,US-020,complete itinerary,semantic


In [12]:
with pd.ExcelWriter(
    input_file,
    engine="openpyxl",
    mode="a",
    if_sheet_exists="replace"
) as writer:

    b_gt_generated.to_excel(
        writer,
        sheet_name="B_GT",
        index=False
    )

print(f"✓ B_GT saved to '{input_file}'")
print(f"✓ B_GT contains {len(b_gt_generated)} rows.")

✓ B_GT saved to 'Ground_truth_OBD_dependency.xlsx'
✓ B_GT contains 542 rows.


In [13]:
unique_pairs = (
    b_gt_generated[["entity_a", "entity_b", "nlp_match_type (exact/semantic/none)"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

output_path = "unique_entity_pairs.csv"
unique_pairs.to_csv(output_path, index=False)

print(f"Unique entity pairs: {len(unique_pairs)}")
print(f"Saved to: {output_path}")

Unique entity pairs: 408
Saved to: unique_entity_pairs.csv


In [14]:
llm_input = b_gt_generated[
    [
        "row_id",
        "story_a_id",
        "action_a",
        "entity_a",
        "story_b_id",
        "action_b",
        "entity_b",
    ]
]

output_path = "llm_input.csv"
llm_input.to_csv(output_path, index=False)

print(f"Rows exported: {len(llm_input)}")
print(f"Saved to: {output_path}")

Rows exported: 542
Saved to: llm_input.csv
